# CattleSense AI — Research Prototype v1.2

### Research question
Can a balanced, feature-extended **Extra Trees** classifier improve rumination recognition and generalize to previously unseen cattle compared with the existing **Random Forest baseline**?

**What this notebook does:** six held-out-cow comparisons; class support and RUS recall; model selection using **validation cows only**; threshold diagnostics; export evidence ZIP. It does **not** update your existing Gradio website.

Dataset: [Japanese Black Beef Cow dataset v2](https://zenodo.org/records/5849025). Do not redistribute the source recordings without checking licensing. These are beef cattle recordings, not dairy-cow field validation.

**Beginner:** Colab → Upload notebook → Runtime → Run all. If download fails, upload `JapaneseBlackBeefData.zip` through left Files tab and rerun. The results ZIP should download in the final cell.

## 1. Setup

In [ ]:
import re, zipfile, urllib.request, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix, recall_score
from IPython.display import display

SEED=42
np.random.seed(SEED)
OUT=Path('cattlesense_v12_results'); OUT.mkdir(exist_ok=True)
KEEP=['RES','RUS','MOV','GRZ']
print('Ready. Results will be saved to', OUT.resolve())

## 2. Download official dataset

In [ ]:
DATA_URL='https://zenodo.org/records/5849025/files/JapaneseBlackBeefData.zip?download=1'
zip_path=Path('JapaneseBlackBeefData.zip')
if not zip_path.exists():
    print('Downloading official ZIP from Zenodo...')
    try:
        urllib.request.urlretrieve(DATA_URL, zip_path)
    except Exception as exc:
        raise RuntimeError('Download failed. Upload JapaneseBlackBeefData.zip manually using the Colab Files sidebar, then retry.') from exc
if not zipfile.is_zipfile(zip_path):
    raise ValueError('The downloaded file is not a valid ZIP. Delete it and download the dataset from Zenodo.')
with zipfile.ZipFile(zip_path) as z:
    csv_names=[name for name in z.namelist() if re.search(r'(?:^|/)cow[1-6]\.csv$',name,re.I)]
assert len(csv_names)==6, f'Expected six cow CSVs; found {csv_names}'
print('ZIP size (MB):', round(zip_path.stat().st_size/1e6,2))
print('Data CSVs:',csv_names)

## 3. Load cows and fix timestamp units

In [ ]:
def normalized(s): return re.sub(r'[^a-z0-9]','',str(s).lower())

def load_cow(file_handle, cow_id):
    df=pd.read_csv(file_handle,low_memory=False)
    norms={normalized(c):c for c in df.columns}
    needed={'x':'accx','y':'accy','z':'accz','label':'label','time':'timestampunix'}
    mapping={new:next((orig for k,orig in norms.items() if k.startswith(prefix)),None) for new,prefix in needed.items()}
    if any(v is None for v in mapping.values()):
        raise ValueError(f'Unrecognized columns in cow {cow_id}: {list(df.columns)}')
    df=df.rename(columns={v:k for k,v in mapping.items()})[['time','x','y','z','label']].copy()
    for col in ['time','x','y','z']:
        df[col]=pd.to_numeric(df[col],errors='coerce')
    # Dataset UNIX timestamps are milliseconds. Subtract origin first to avoid float precision loss.
    df['time']=(df['time'].astype('float64')-float(df['time'].iloc[0]))/1000.0
    df['label']=df['label'].astype(str).str.strip().str.upper()
    df['cow']=cow_id
    return df

all_cows={}
with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        cow=int(re.search(r'cow([1-6])\.csv$',name,re.I).group(1))
        with z.open(name) as f: all_cows[cow]=load_cow(f,cow)
print('Raw samples / cow:',{c:len(d) for c,d in all_cows.items()})
print('All labels:',sorted(set().union(*(set(d.label.unique()) for d in all_cows.values()))))
for cow,df in sorted(all_cows.items()):
    diffs=df.time.diff().dropna()
    positive=diffs[diffs>0]
    print(f'Cow {cow}: positive timestamp delta (median) = {positive.median() if len(positive) else "unavailable"}')

## 4. Activity distribution (supports/limitations)

In [ ]:
dist=pd.DataFrame({f'cow_{k}':v.label.value_counts() for k,v in all_cows.items()}).fillna(0).astype(int)
display(dist.reindex(KEEP,fill_value=0))
ax=dist.reindex(KEEP,fill_value=0).T.plot.bar(figsize=(9,4),title='Number of labeled raw samples by cow')
ax.set_ylabel('Samples'); plt.tight_layout(); plt.savefig(OUT/'class_distribution.png',dpi=160); plt.show()

## 5. Valid motion windows and engineered features

Baseline uses the same **17 features** as v1.1. Extended variant uses those 17 plus 10 additional signal features derived from the **same** 50-sample window. Windows never cross time gaps and never mix ground-truth labels. The extra features may or may not improve performance.

In [ ]:
FEATURES_BASE = ['acc_mean_x','acc_mean_y','acc_mean_z','acc_std_x','acc_std_y','acc_std_z',
 'acc_min_x','acc_min_y','acc_min_z','acc_max_x','acc_max_y','acc_max_z',
 'acc_energy_x','acc_energy_y','acc_energy_z','magnitude_mean','magnitude_std']
FEATURES_EXTRA = ['median_x','median_y','median_z','mean_abs_diff_x','mean_abs_diff_y','mean_abs_diff_z',
                  'magnitude_median','magnitude_range','magnitude_mean_abs_diff','magnitude_energy']

def base_features(a):
    a=np.asarray(a,dtype=np.float32)
    mag=np.linalg.norm(a,axis=1)
    return np.r_[a.mean(axis=0), a.std(axis=0), a.min(axis=0), a.max(axis=0),
                 (a*a).mean(axis=0), mag.mean(),mag.std()].astype(np.float32)

def extended_features(a):
    a=np.asarray(a,dtype=np.float32)
    mag=np.linalg.norm(a,axis=1)
    d=np.diff(a,axis=0)
    dm=np.diff(mag)
    extra=np.r_[np.median(a,axis=0),np.abs(d).mean(axis=0),
                 np.median(mag),np.ptp(mag),np.abs(dm).mean(),np.mean(mag**2)]
    return np.r_[base_features(a),extra].astype(np.float32)

def make_windows(df,width=50,stride=25,max_gap_seconds=0.12):
    a=df[['x','y','z']].to_numpy(dtype=np.float32)
    times=df.time.to_numpy(dtype=np.float64)
    labels=df.label.to_numpy()
    xb,xe,ys=[],[],[]
    rejected={'label':0,'missing':0,'time_gap':0}
    for start in range(0,len(df)-width+1,stride):
        target=labels[start:start+width]
        data=a[start:start+width]
        t=times[start:start+width]
        if target[0] not in KEEP or not np.all(target==target[0]):
            rejected['label']+=1; continue
        if not (np.isfinite(data).all() and np.isfinite(t).all()):
            rejected['missing']+=1; continue
        dt=np.diff(t)
        if np.any(dt<=0) or np.any(dt>max_gap_seconds):
            rejected['time_gap']+=1; continue
        xb.append(base_features(data));xe.append(extended_features(data));ys.append(target[0])
    return (np.asarray(xb,dtype=np.float32).reshape(-1,17),
            np.asarray(xe,dtype=np.float32).reshape(-1,27),np.asarray(ys),rejected)

windows={}
for cow,df in sorted(all_cows.items()):
    Xb,Xe,y,rej=make_windows(df)
    windows[cow]={'base':Xb,'extended':Xe,'y':y}
    print(f'Cow {cow}: {len(y)} windows | classes={dict(pd.Series(y).value_counts())} | excluded={rej}')
    if len(y)==0: raise ValueError(f'No usable windows for cow {cow}.')
print('Feature lengths:',len(FEATURES_BASE),len(FEATURES_BASE+FEATURES_EXTRA))

## 6. Fair six-cow model comparison

Each round: one test cow, one **different** validation cow, remaining four training cows. Same splits and same windows for both candidates.

- **Baseline:** v1.1-style Random Forest, 17 features.
- **Candidate:** Extra Trees, 27 features, balanced class weights.

The test cow never influences training or model choice. We select the candidate **using validation macro F1** then report its untouched test performance. Because the dataset is small and some cows lack classes, treat all results as preliminary.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

LABELS=KEEP
rows=[]
per_class=[]
all_predictions=[]

def stack(cows,field):
    return (np.concatenate([windows[i][field] for i in cows]),
            np.concatenate([windows[i]['y'] for i in cows]))

def build_model(name):
    if name=='Random Forest (17)':
        return RandomForestClassifier(n_estimators=150,max_depth=18,
            class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
    return ExtraTreesClassifier(n_estimators=200,min_samples_leaf=2,
        class_weight='balanced',random_state=SEED,n_jobs=-1)

for test_cow in sorted(windows):
    others=[i for i in sorted(windows) if i!=test_cow]
    val_cow=others[0]
    train_cows=[i for i in others if i!=val_cow]
    print('\nTesting cow',test_cow,'| validation cow',val_cow,'| training cows',train_cows)
    for name,field in [('Random Forest (17)','base'),('Extra Trees (27)','extended')]:
        Xtr,ytr=stack(train_cows,field)
        Xval,yval=stack([val_cow],field)
        Xt,yt=stack([test_cow],field)
        model=build_model(name)
        model.fit(Xtr,ytr)
        pred_val=model.predict(Xval)
        pred=model.predict(Xt)
        val_f1=float(f1_score(yval,pred_val,labels=LABELS,average='macro',zero_division=0))
        acc=float(accuracy_score(yt,pred))
        f1=float(f1_score(yt,pred,labels=LABELS,average='macro',zero_division=0))
        true_rus=int(np.sum(yt=='RUS'))
        rus_recall=float(np.mean(pred[yt=='RUS']=='RUS')) if true_rus else None
        probs=model.predict_proba(Xt)
        conf=probs.max(axis=1)
        predicted_by_prob=model.classes_[probs.argmax(axis=1)]
        rows.append({'test_cow':test_cow,'validation_cow':val_cow,
            'train_cows':','.join(map(str,train_cows)),'model':name,
            'test_windows':len(yt),'test_classes':len(set(yt)),
            'val_macro_f1_4_classes':val_f1, 'test_accuracy':acc,
            'test_macro_f1_4_classes':f1,'rus_actual_support':true_rus,
            'rus_recall':rus_recall})
        for label in LABELS:
            mask=(yt==label)
            per_class.append({'test_cow':test_cow,'model':name,'label':label,
               'actual_support':int(mask.sum()),
               'recall':float(np.mean(pred[mask]==label)) if mask.any() else None})
        # Diagnostic thresholds are fixed in advance, never tuned on test animals.
        for threshold in [0.5,0.7,0.9]:
            take=conf>=threshold
            all_predictions.append({'test_cow':test_cow,'model':name,'threshold':threshold,
                'coverage':float(take.mean()),'accepted_count':int(take.sum()),
                'accepted_accuracy':float(np.mean(predicted_by_prob[take]==yt[take])) if take.any() else None})
        print(f'  {name}: accuracy {acc:.1%}; macro F1 {f1:.1%}; RUS recall {rus_recall if rus_recall is not None else "N/A"}')

comparison=pd.DataFrame(rows)
class_results=pd.DataFrame(per_class)
thresholds=pd.DataFrame(all_predictions)
comparison.to_csv(OUT/'model_comparison_by_cow.csv',index=False)
class_results.to_csv(OUT/'class_recall_by_cow.csv',index=False)
thresholds.to_csv(OUT/'confidence_thresholds.csv',index=False)
display(comparison.round(4))

## 7. Compare models and avoid cherry-picking

The summary includes a **validation-selected winner per cow**. This is a fairer prospective policy than selecting the model with the best test score, but the tiny validation sample is a limitation. The baseline model is still the reference; do not replace your website until we review the results.

In [ ]:
means=comparison.groupby('model',as_index=False).agg(
    mean_accuracy=('test_accuracy','mean'),
    mean_macro_f1=('test_macro_f1_4_classes','mean'),
    mean_rus_recall_among_supported_cows=('rus_recall','mean'))
print('Six held-out-cow means:')
display(means.round(4))

chosen=(comparison.sort_values(['test_cow','val_macro_f1_4_classes','model'],
    ascending=[True,False,True]).groupby('test_cow',as_index=False).head(1).sort_values('test_cow'))
chosen.to_csv(OUT/'validation_selected_model_per_cow.csv',index=False)
print('Selected from validation (NOT from test accuracy):')
display(chosen[['test_cow','model','val_macro_f1_4_classes','test_accuracy','test_macro_f1_4_classes','rus_recall']].round(4))

plotdata=comparison.pivot(index='test_cow',columns='model',values='test_macro_f1_4_classes')
ax=plotdata.plot.bar(figsize=(9,4),ylim=(0,1),title='Held-out-cow Macro F1 (4 target classes)')
ax.set_ylabel('Macro F1');ax.set_xlabel('Test cow');plt.tight_layout()
plt.savefig(OUT/'model_comparison.png',dpi=170);plt.show()

r=class_results[class_results.label=='RUS'].pivot(index='test_cow',columns='model',values='recall')
ax=r.plot.bar(figsize=(9,4),ylim=(0,1),title='Ruminating (RUS) recall — where labeled RUS exists')
ax.set_ylabel('Recall');ax.set_xlabel('Test cow');plt.tight_layout()
plt.savefig(OUT/'rumination_recall.png',dpi=170);plt.show()

print('Note: missing bars reflect no RUS examples, not zero recall.')

## 8. Confidence policy: fixed thresholds for research diagnostics

Do not claim that thresholded predictions are calibrated probabilities. At high thresholds, coverage may become too low to be useful. This section compares accepted accuracy **and** coverage.

In [ ]:
agg=thresholds.groupby(['model','threshold'],as_index=False).agg(
    mean_coverage=('coverage','mean'),
    mean_accepted_accuracy=('accepted_accuracy','mean'))
display(agg.round(4))
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,sub in agg.groupby('model'):
    axes[0].plot(sub.threshold,sub.mean_coverage,marker='o',label=name)
    axes[1].plot(sub.threshold,sub.mean_accepted_accuracy,marker='o',label=name)
axes[0].set(title='Average coverage by threshold',xlabel='Confidence cutoff',ylabel='Coverage',ylim=(0,1.05))
axes[1].set(title='Accepted accuracy by threshold',xlabel='Confidence cutoff',ylabel='Accepted accuracy',ylim=(0,1.05))
for ax in axes: ax.grid(alpha=.2);ax.legend(fontsize=8)
plt.tight_layout();plt.savefig(OUT/'confidence_diagnostics.png',dpi=160);plt.show()

## 9. Export research files

The resulting ZIP contains only derived metrics, charts and notes. **It does not contain a model for deployment**, because this is a controlled comparison stage. We will choose the next deployment candidate after examining the results and preserving a locked test protocol.

In [ ]:
from zipfile import ZipFile
summary={
 'experiment':'CattleSense AI v1.2 cross-animal model comparison',
 'dataset':'Japanese Black Beef Cow Behavior Classification Dataset v2',
 'dataset_link':'https://zenodo.org/records/5849025',
 'models':['Random Forest (17)','Extra Trees (27)'],
 'six_held_out_cows':list(sorted(windows)),
 'features_baseline':FEATURES_BASE,
 'features_candidate':FEATURES_BASE+FEATURES_EXTRA,
 'means':means.to_dict(orient='records'),
 'validation_selected_average_accuracy':float(chosen.test_accuracy.mean()),
 'validation_selected_average_macro_f1':float(chosen.test_macro_f1_4_classes.mean()),
 'note':'Only six beef cows; overlapping windows; missing per-animal classes; thresholds are diagnostics, not calibration.'
}
(OUT/'research_summary_v12.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'LIMITATIONS.txt').write_text(
 'Small sample of six beef cattle, not dairy field deployment.\n'
 'Animal-held-out testing prevents same-cow train/test overlap but not farm or sensor transfer effects.\n'
 'Some cows have no RUS examples. Missing recall is not zero.\n'
 'Class selection based on validation can be unstable on a single cow.\n'
 'Window-level metrics are correlated because windows overlap.\n'
 'No hardware, power or energy evaluations. No model deployed in this stage.\n',encoding='utf-8')
zipname='CattleSense_v1_2_Research_Results.zip'
with ZipFile(zipname,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()): z.write(f,f.name)
print('Ready:',zipname)
try:
    from google.colab import files
    files.download(zipname)
except ImportError:
    print('Download ZIP manually from notebook working directory.')